# Handoff Customization

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use the `handoff()` function for fine-grained control over agent transfers: filtering history with `input_filter`, requiring structured data with `input_type`, running callbacks with `on_handoff`, and renaming tools with `tool_name_override`.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic handoff() Function

Use `handoff()` instead of passing agents directly to `handoffs=[]` to enable customization.

In [ ]:
from agents import Agent, Runner, handoff

billing_agent = Agent(
    name="Billing Agent",
    instructions="You handle billing and payment questions.",
)

support_agent = Agent(
    name="Support Agent",
    instructions="You are front-line support. Hand off billing questions to the billing specialist.",
    handoffs=[handoff(billing_agent)],
)

result = await Runner.run(support_agent, "I have a question about my invoice.")
print(f"Final agent: {result.last_agent.name}")
print(result.final_output)

## 4. Filtering History with input_filter

Use `handoff_filters.remove_all_tools` to strip tool call messages from the history, or write a custom filter.

In [ ]:
from agents import handoff_filters

specialist = Agent(
    name="Specialist",
    instructions="You handle complex technical issues based on the conversation context.",
)

triage = Agent(
    name="Triage",
    instructions="Route complex issues to the specialist.",
    handoffs=[handoff(
        specialist,
        input_filter=handoff_filters.remove_all_tools,
    )],
)

result = await Runner.run(triage, "I have a complex technical issue with my API integration.")
print(f"Final agent: {result.last_agent.name}")
print(result.final_output)

## 5. Custom Input Filter

Write your own filter function to control exactly what the receiving agent sees.

In [ ]:
def keep_last_n_messages(handoff_input):
    handoff_input.history = handoff_input.history[-6:]
    return handoff_input


specialist_v2 = Agent(
    name="Specialist",
    instructions="You handle escalated issues.",
)

triage_v2 = Agent(
    name="Triage",
    instructions="Route to the specialist for complex issues.",
    handoffs=[handoff(
        specialist_v2,
        input_filter=keep_last_n_messages,
    )],
)

result = await Runner.run(triage_v2, "I need help with a complex database migration issue.")
print(result.final_output)

## 6. Structured Handoff Data with input_type

Require the handing-off agent to provide structured data via a Pydantic model.

In [ ]:
from pydantic import BaseModel


class EscalationData(BaseModel):
    reason: str
    priority: str
    customer_id: str


escalation_agent = Agent(
    name="Escalation Handler",
    instructions="You handle escalated issues. Use the escalation context provided.",
)

support = Agent(
    name="Support Agent",
    instructions=(
        "You handle support requests. When escalating, provide the reason, "
        "priority (low/medium/high), and customer ID (use 'cust_123')."
    ),
    handoffs=[handoff(
        escalation_agent,
        input_type=EscalationData,
    )],
)

result = await Runner.run(support, "I've been waiting 3 days for a response. This is urgent!")
print(f"Final agent: {result.last_agent.name}")
print(result.final_output)

## 7. Callbacks with on_handoff

Run custom logic when a handoff occurs — useful for logging, analytics, or side effects.

In [ ]:
async def log_handoff(ctx, handoff_input):
    print(f"[LOG] Handoff occurred with input: {handoff_input}")


billing = Agent(
    name="Billing Agent",
    instructions="You handle billing questions.",
)

front_desk = Agent(
    name="Front Desk",
    instructions="Route billing questions to billing.",
    handoffs=[handoff(
        billing,
        on_handoff=log_handoff,
    )],
)

result = await Runner.run(front_desk, "I need to understand my latest charge.")
print(result.final_output)

## 8. Renaming the Handoff Tool

Use `tool_name_override` to give the handoff tool a custom name.

In [ ]:
escalation_handler = Agent(
    name="Escalation Handler",
    instructions="You handle urgent escalations.",
)

support_v3 = Agent(
    name="Support Agent",
    instructions="Use the escalate tool for urgent issues.",
    handoffs=[handoff(
        escalation_handler,
        tool_name_override="escalate",
    )],
)

result = await Runner.run(support_v3, "This is an emergency! Our production system is down!")
print(f"Final agent: {result.last_agent.name}")
print(result.final_output)

## 9. Combining All Customizations

All handoff options can be combined for maximum control.

In [ ]:
class HandoffData(BaseModel):
    summary: str
    urgency: str


async def on_escalate(ctx, data: HandoffData):
    print(f"[LOG] Escalation: {data.summary} [urgency: {data.urgency}]")


specialist_final = Agent(name="Specialist", instructions="Handle escalated issues.")

triage_final = Agent(
    name="Triage Agent",
    instructions=(
        "Triage incoming requests. Escalate when needed with a summary and urgency level "
        "(low/medium/high)."
    ),
    handoffs=[handoff(
        specialist_final,
        tool_name_override="escalate",
        input_type=HandoffData,
        input_filter=handoff_filters.remove_all_tools,
        on_handoff=on_escalate,
    )],
)

result = await Runner.run(triage_final, "Our API is returning 500 errors and customers are affected.")
print(result.final_output)

## Key Takeaways

- `handoff()` provides fine-grained control over agent-to-agent transfers
- `input_filter` controls what conversation history the receiving agent sees
- `handoff_filters.remove_all_tools` strips tool messages to reduce noise
- `input_type` requires structured data (Pydantic model) when initiating a handoff
- `on_handoff` runs custom callbacks for logging or side effects
- `tool_name_override` renames the handoff tool for clearer agent instructions